In [14]:
print("can anyone hear me?")

can anyone hear me?


In [15]:
#setup
%matplotlib inline
from __future__ import print_function
from six.moves import zip, range
import pandas as pd
import numpy as np
# import jellyfish
# from collections import OrderedDict
import splink as sp
from splink import DuckDBAPI, Linker, SettingsCreator, block_on, splink_datasets
db_api=DuckDBAPI()

In [16]:
##Linking!!
import splink.comparison_library as cl
df_princelings_lt = pd.read_csv('/Users/maggiewang/Downloads/Year_4/Thesis/dataverse_files/coded_princelings.csv')
df_complete_lt = pd.read_csv('/Users/maggiewang/Downloads/Year_4/Thesis/coded_complete.csv')
from splink.comparison_library import ExactMatch
settings = SettingsCreator(
    link_type="link_only",
    unique_id_column_name = "unique_id",
    probability_two_random_records_match = 1/1208621,
    blocking_rules_to_generate_predictions=[
        block_on('lnarea'),
        block_on('lnprice'),
        block_on('land_grade'),
        block_on('province'),
        block_on ('date')],

    additional_columns_to_retain=['unique_id', 'lnarea', 'lnprice', 'land_grade', 'province'],
    
    comparisons = [
        cl.ExactMatch('lnarea').configure(
            term_frequency_adjustments=True,),
        cl.ExactMatch('lnprice').configure(
            term_frequency_adjustments=True,),
        cl.ExactMatch('land_grade').configure(
            term_frequency_adjustments=True,),
        cl.ExactMatch('province').configure(
            term_frequency_adjustments=True,),
        cl.ExactMatch('date').configure(
            term_frequency_adjustments=True,),
], 
)

linker = Linker([df_complete_lt, df_princelings_lt], settings, db_api)

# splink_df = linker.table_management.register_table("table1", "mergedlt")
# splink_df.as_pandas_dataframe()

In [18]:
linker.training.estimate_u_using_random_sampling(max_pairs=1e8)
training_features = block_on('land_grade', 'province', 'lnarea')
training_session_features = linker.training.estimate_parameters_using_expectation_maximisation(training_features)

----- Estimating u probabilities using random sampling -----

Estimated u probabilities using random sampling

Your model is not yet fully trained. Missing estimates for:
    - land_grade (no m values are trained).
    - province (no m values are trained).
    - date (no m values are trained).

----- Starting EM training session -----

Estimating the m probabilities of the model by blocking on:
(l."land_grade" = r."land_grade") AND (l."province" = r."province") AND (l."lnarea" = r."lnarea")

Parameter estimates will be made for the following comparison(s):
    - date

Parameter estimates cannot be made for the following comparison(s) since they are used in the blocking rules: 
    - land_grade
    - province

Level Exact match on date on comparison date not observed in dataset, unable to train m value

Iteration 1: Largest change in params was 0.95 in the m_probability of date, level `All other comparisons`
Iteration 2: Largest change in params was 8.43e-10 in probability_two_random_re

In [17]:
##train model to estimate parameters
linker.training.estimate_u_using_random_sampling(max_pairs=1e8)
training_aprice = block_on('lnarea', 'lnprice')
training_session_aprice = linker.training.estimate_parameters_using_expectation_maximisation(training_aprice)

----- Estimating u probabilities using random sampling -----

Estimated u probabilities using random sampling

Your model is not yet fully trained. Missing estimates for:
    - lnarea (no m values are trained).
    - lnprice (no m values are trained).
    - land_grade (no m values are trained).
    - province (no m values are trained).
    - date (no m values are trained).

----- Starting EM training session -----

Estimating the m probabilities of the model by blocking on:
(l."lnarea" = r."lnarea") AND (l."lnprice" = r."lnprice")

Parameter estimates will be made for the following comparison(s):
    - land_grade
    - province
    - date

Parameter estimates cannot be made for the following comparison(s) since they are used in the blocking rules: 
    - lnarea
    - lnprice


EMTrainingException: Training rule `(l."lnarea" = r."lnarea") AND (l."lnprice" = r."lnprice")` resulted in no record pairs.  This means that in the supplied data set there were no pairs of records for which `(l."lnarea" = r."lnarea") AND (l."lnprice" = r."lnprice")` was `true`.
Expectation maximisation requires a substantial number of record comparisons to produce accurate parameter estimates - usually at least a few hundred, but preferably at least a few thousand.
You must revise your training blocking rule so that the set of generated comparisons is not empty.  You can use `linker.count_num_comparisons_from_blocking_rule()` to compute the number of comparisons that will be generated by a blocking rule.

In [ ]:
##visuals
linker.evaluation.unlinkables_chart()

In [ ]:
linker.visualisations.parameter_estimate_comparisons_chart()

In [ ]:
linker.visualisations.m_u_parameters_chart()

In [ ]:
#saving the model
settings = linker.misc.save_model_to_json(
    "LTmodel.json", overwrite=True
)

